In [1]:
import json, os
import numpy as np
import torch
#from torch.utils.data import Dataset

In [2]:
import sys
sys.path.append('../')
sys.path.append('./')
# sys.path.append('../preprocess/')
# sys.path.append('../evaluation/')
# sys.path.append('preprocess/') 
# sys.path.append('evaluation/')
print(sys.path)

['/mount/arbeitsdaten31/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/intoxicat', '/usr/lib64/python310.zip', '/usr/lib64/python3.10', '/usr/lib64/python3.10/lib-dynload', '', '/mount/arbeitsdaten31/studenten1/team-lab-phonetics/2023/student_directories/kolos/NLP4/lib64/python3.10/site-packages', '/mount/arbeitsdaten31/studenten1/team-lab-phonetics/2023/student_directories/kolos/NLP4/lib/python3.10/site-packages', '../', './']


In [3]:
from preprocess.prepare_data import split_dataset_into_splits
from preprocess.data_utilities import get_keep_features, Dataset, collate_costum
# from data_utility import get_keep_features, Dataset, collate_costum
# from prepare_data import split_dataset_into_splits

In [4]:
!pwd

/mount/arbeitsdaten31/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/intoxicat


In [9]:
folder_name = r'preprocess/dl2'
features = 'LLD' #'Functional'

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [7]:
path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/'
#subset_name = 'V2_bal_subset_functional_type_full.json'
subset_name = 'V2_bal_subset_LLD_type_full.json'

path_to_subset = os.path.join(path, subset_name)

In [ ]:
#split_dataset_into_splits([path_to_subset], 'functional', folder_name)
split_dataset_into_splits([path_to_subset], 'LLD', folder_name)

In [10]:
print(path_to_subset)
short_path_to_subset = os.path.splitext(path_to_subset)[0]
short_subset_name = os.path.splitext(subset_name)[0]
final_path_to_subset = os.path.join(folder_name, short_subset_name)
print(short_subset_name)

/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/V2_bal_subset_LLD_type_full.json
V2_bal_subset_LLD_type_full


In [11]:
train_dataset = Dataset('{}_train.json'.format(final_path_to_subset, features), features)
valid_dataset = Dataset('{}_valid.json'.format(final_path_to_subset, features), features)
test_dataset = Dataset('{}_test.json'.format(final_path_to_subset, features), features)

In [12]:
train_dataset.feature_names[:10]

['F0semitoneFrom27.5Hz_sma3nz',
 'F1amplitudeLogRelF0_sma3nz',
 'F1bandwidth_sma3nz',
 'F1frequency_sma3nz',
 'F2amplitudeLogRelF0_sma3nz',
 'F2bandwidth_sma3nz',
 'F2frequency_sma3nz',
 'F3amplitudeLogRelF0_sma3nz',
 'F3bandwidth_sma3nz',
 'F3frequency_sma3nz']

In [13]:
len(train_dataset.feature_names)

25

In [14]:
print(f'''
train_dataset: {len(train_dataset)} samples
valid_dataset: {len(valid_dataset)} samples
test_dataset: {len(test_dataset)} samples
''') #for balanced: use test and validate directly from the corresponding dir


train_dataset: 1283 samples
valid_dataset: 160 samples
test_dataset: 161 samples



In [15]:
#train_dataset.labels = torch.tensor([x[0] for x in train_dataset.labels])
#valid_dataset.labels = torch.tensor([x[0] for x in valid_dataset.labels])
#test_dataset.labels = torch.tensor([x[0] for x in test_dataset.labels])

In [52]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, num_heads, dropout):
        super(TransformerNet, self).__init__()
        
        self.embedding = nn.Linear(input_dim, hidden_dim)
        self.encoder_layers = nn.TransformerEncoderLayer(hidden_dim, num_heads, dim_feedforward=hidden_dim*4, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layers, num_layers)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
#         print(x.shape) #torch.Size([32, min_VAR, 25])
        #input: batch, seqlength=1, feature length
        x = self.embedding(x)
        x = x.permute(1, 0, 2)  # Reshape to (sequence_length, batch_size, hidden_dim)
        x = self.transformer_encoder(x)
        x = x.permute(1, 0, 2)  # Reshape back to (batch_size, sequence_length, hidden_dim)
        x = x.mean(dim=1)  # Average pooling over the sequence length
        x = self.fc(x)
        return F.softmax(x, dim=1) #log_softmax
    

In [53]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
def collate_costum1(batch):
    
    
    # batch is a list of tuples with labels, file features and lengths, and file names
    #batch features: 32 * 25 * VAR. feature_length_list contains these VAR lengths for each time sequence
    
    label_list, feature_list, feature_length_list, file_name_list = [], [], [], []
    
    # iterate over batch
    # each element represents a file
    for element in batch:
        # add to lists for each element
        label_list.append(element[0])
        feature_list.append(element[1])
        feature_length_list.append(element[2])
        file_name_list.append(element[3])
    
#     print(f'Got {len(feature_list)} elements in feature_list')
    
    lengths_of_features = set([len(x) for x in feature_list])
#     print(lengths_of_features)
#     print('feature_list[0].shape before truncation')
#     print(feature_list[0].shape)
#     print(feature_length_list) 
    
    min_feature_length = min(feature_length_list) - 1
#     print(f'Features will be reduced to {min_feature_length}')
    
    feature_list = torch.stack([torch.stack([torch.Tensor(y[:min_feature_length]) for y in x]) 
                                 for x in feature_list])

#     print(len(feature_list))
#     print('feature_list[0].shape after truncation')
#     print(feature_list[0].shape)
#     print('Resulting feature_list.shape:')
#     print(feature_list.shape)
    
        
        
    # stack the label one hot encodings
    stacked_labels = torch.stack(label_list)

#     # get number of features 
#     number_of_features = len(feature_list[0])
#     # flatten current feature list so that it becomes a list of list 
#     # instead of a list of lists of matrices
#     # this is necessary in order to use the padding function
#     flattened_feature_list = [feature_tensor for file_features in feature_list for feature_tensor in file_features]
#     # use torch's padding function on flattened list
#     flattened_stacked_features = pad_sequence(flattened_feature_list, batch_first=True)
#     # transform the padded flattened tensor to a tensor of matrices
#     # number of features indicate how many rows belong to one matrix (i.e. file)
#     # permute shape of tensor to make it accessible for the model
#     stacked_features = torch.permute(torch.stack(torch.split(flattened_stacked_features, number_of_features)), (0, 2, 1))
    stacked_features = torch.permute(feature_list, (0, 2, 1))
#     print(feature_list.shape) #torch.Size([16, 25, 269])
    
    # why do we need the length?
    stacked_feature_lengths = torch.LongTensor(feature_length_list)

    return stacked_labels, stacked_features, stacked_feature_lengths, file_name_list

In [54]:
batch_size = 16#32
a = torch.utils.data.DataLoader(train_dataset, collate_fn=collate_costum1, batch_size=batch_size, drop_last=True)
for idx, batch in enumerate(a):
    print(f'This was batch {idx}')
    if idx > 1: 
        break

This was batch 0
This was batch 1
This was batch 2


In [55]:
import torch.optim as optim

# Define hyperparameters
input_dim = 25 #88 -- Functional, 25 -- LLD
hidden_dim = 256
output_dim = 2 #1
num_layers = 4
num_heads = 4
dropout = 0.2
learning_rate = 0.001
batch_size = 32
num_epochs = 10

train_loader = torch.utils.data.DataLoader(train_dataset, collate_fn=collate_costum1, batch_size=batch_size, drop_last=True)

# Create an instance of the TransformerNet model
model = TransformerNet(input_dim, hidden_dim, output_dim, num_layers, num_heads, dropout)

# Define the loss function and optimizer
criterion = nn.BCELoss() #replace with sth for binary target
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


# Assuming you have your training data in `train_loader`
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for batch_idx, (target, features, _, _) in enumerate(train_loader):
#         print(f'{len(target)} examples: ', features.shape)
        optimizer.zero_grad()
        
        # Forward pass
        output = model(features)
        
#         print(output.shape)
#         print(target.shape)
        
        target = target.type('torch.FloatTensor')
        
        # Compute the loss
        loss = criterion(output, target)
        total_loss += loss.item()
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Print batch loss every few batches
        if (batch_idx + 1) % 10 == 0:
            print('Epoch [{}/{}], Batch [{}/{}], Loss: {:.4f}'
                  .format(epoch + 1, num_epochs, batch_idx + 1, len(train_loader), loss.item()))
    
    # Print epoch loss
    print('Epoch [{}/{}], Loss: {:.4f}'.format(epoch + 1, num_epochs, total_loss / len(train_loader)))

Epoch [1/10], Batch [10/40], Loss: 0.8558
Epoch [1/10], Batch [20/40], Loss: 0.6789
Epoch [1/10], Batch [30/40], Loss: 0.6706
Epoch [1/10], Batch [40/40], Loss: 0.7011
Epoch [1/10], Loss: 0.9094
Epoch [2/10], Batch [10/40], Loss: 0.6997
Epoch [2/10], Batch [20/40], Loss: 0.6883
Epoch [2/10], Batch [30/40], Loss: 0.6730
Epoch [2/10], Batch [40/40], Loss: 0.7063
Epoch [2/10], Loss: 0.6995
Epoch [3/10], Batch [10/40], Loss: 0.6965
Epoch [3/10], Batch [20/40], Loss: 0.6819
Epoch [3/10], Batch [30/40], Loss: 0.6679
Epoch [3/10], Batch [40/40], Loss: 0.7070
Epoch [3/10], Loss: 0.7000
Epoch [4/10], Batch [10/40], Loss: 0.7008
Epoch [4/10], Batch [20/40], Loss: 0.6838
Epoch [4/10], Batch [30/40], Loss: 0.6688
Epoch [4/10], Batch [40/40], Loss: 0.7050
Epoch [4/10], Loss: 0.7008
Epoch [5/10], Batch [10/40], Loss: 0.6992
Epoch [5/10], Batch [20/40], Loss: 0.6836
Epoch [5/10], Batch [30/40], Loss: 0.6684
Epoch [5/10], Batch [40/40], Loss: 0.7070
Epoch [5/10], Loss: 0.7005
Epoch [6/10], Batch [10/4

In [64]:
def postpr_output(output):
    print(output.shape)
    return torch.tensor([torch.argmax(x) for x in output])

def evaluate(model, valid_loader):
    model.eval()
    total_loss = 0
    for batch_idx, (target, features, _, _) in enumerate(valid_loader):
#         print(target)
#         print()
#         print(features)
        # Forward pass
        output = model(features)
        
        target = target.type('torch.FloatTensor')
        
        # Compute the loss
        loss = criterion(output, target)
        total_loss += loss.item()
        
        # Compute the accuracy
        output = torch.argmax(output, dim=1)
        target = torch.argmax(target, dim=1)
        
        n_correct = torch.sum(output == target)
        acc = n_correct/len(target)
        
        # Print batch loss every few batches
        if (batch_idx + 1) % 2 == 0:
            print('Batch [{}/{}], Loss: {:.4f}, Acc: {:.4f}'
                  .format(batch_idx + 1, len(valid_loader), loss.item(), acc))
    print('Loss: {:.4f}'.format(total_loss / len(valid_loader)))

In [65]:
model.eval()
valid_loader = torch.utils.data.DataLoader(valid_dataset, collate_fn=collate_costum1, batch_size=batch_size, drop_last=True)
evaluate(model, valid_loader)

Batch [2/5], Loss: 0.7154, Acc: 0.2812
Batch [4/5], Loss: 0.6943, Acc: 0.5000
Loss: 0.6973


In [66]:
model.eval()
test_loader = torch.utils.data.DataLoader(test_dataset, collate_fn=collate_costum1, batch_size=batch_size, drop_last=True)
evaluate(model, test_loader)

Batch [2/5], Loss: 0.6943, Acc: 0.5000
Batch [4/5], Loss: 0.6883, Acc: 0.5625
Loss: 0.6907


# Sandbox Below

In [67]:
item_ = 0
for batch_idx, item in enumerate(train_loader):
    item_ = item
    print(batch_idx)
    break

0


In [68]:
item_[0][:10] #sober, intoxicated

tensor([[0, 1],
        [1, 0],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [1, 0],
        [0, 1],
        [0, 1]])

In [85]:
x = item_[1]
input_dim = 88
hidden_dim = 10

fc = nn.Linear(input_dim, hidden_dim)
#fc(x)
output = model(x)
target = item_[0]

In [101]:
def postpr_output(output):
    print(output.shape)
    return torch.tensor([torch.argmax(x) for x in output])

postpr_output(output)

torch.Size([32, 2])


tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0])

In [88]:
nn.BCELoss()(output, target.type('torch.FloatTensor'))

tensor(0.8550, grad_fn=<BinaryCrossEntropyBackward0>)